# Import

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import config
from data_loader import DataLoader
from preprocessing import (
    shuffle_data, encode_labels, split_data,
    normalize, get_class_weights
)
from model import build_cnn, show_architecture

# Load Dataset

In [2]:
loader = DataLoader(config.DATA_PATH, target_size=config.IMAGE_SIZE)
images, labels = loader.load_all_images()
X, y = loader.to_numpy_arrays(images, labels)

class_names = loader.classes
print("Classes:", class_names)


Found classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Target image size: (224, 224)

Loading images...
  Class 'glioma': 1621 images loaded
  Class 'meningioma': 1645 images loaded
  Class 'notumor': 2000 images loaded
  Class 'pituitary': 1757 images loaded

Total images loaded: 7023

Numpy arrays created:
  X shape: (7023, 224, 224, 3)
  y shape: (7023,)
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


# shuffle images 

In [3]:
X,y = shuffle_data(X,y)

data shuffled (7023 samples)


In [4]:
print(f"first 10 labels: {y[:10]}")

first 10 labels: [1 2 3 2 3 0 3 1 0 1]


# Encode Labels (one-hot)

In [5]:
y_onehot,encoder = encode_labels(y,num_classes=config.NUM_CLASSES)
print(f"Example: {y[0]} -> {y_onehot[0]}")

labels encoded: [0, 1, 2, 3]
shape: (7023, 4)
Example: 1 -> [0. 1. 0. 0.]


# Split train / val / test

In [6]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X, y_onehot,
    test_size=config.TEST_SIZE,
    val_size=config.VAL_SIZE
)


Data split:
  Train      : 4915 images
  Validation : 703 images
  Test       : 1405 images


# Normalize pixels to [0, 1]

In [7]:
X_train = normalize(X_train)
X_val   = normalize(X_val)
X_test  = normalize(X_test)

print("\nFinal shapes:")
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "| y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "| y_test :", y_test.shape)




Normalized => min=0.0, max=1.0
Normalized => min=0.0, max=1.0
Normalized => min=0.0, max=1.0

Final shapes:
X_train: (4915, 224, 224, 3) | y_train: (4915, 4)
X_val  : (703, 224, 224, 3) | y_val  : (703, 4)
X_test : (1405, 224, 224, 3) | y_test : (1405, 4)


# Class weights (handle imbalance)

In [8]:
y_train_int = y_train.argmax(axis=1)
class_weights = get_class_weights(y_train_int)


class weights: {0: 1.0825991189427313, 1: 1.0675499565595135, 2: 0.8776785714285714, 3: 0.9997965825874695}


# Build the CNN

In [ ]:
import sys
import pydot

print("Python:", sys.executable)
print("pydot version:", pydot.__version__)
print("pydot location:", pydot.__file__)
print("Has InvocationException:",
      hasattr(pydot, "InvocationException"))

In [10]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.13.0
Keras: 2.13.1


In [11]:

import os
import shutil

print("dot found in PATH:", shutil.which("dot"))
print(
    "Graphviz executable exists:",
    os.path.exists(r"C:\Program Files (x86)\Graphviz\bin\dot.exe")
)


dot found in PATH: None
Graphviz executable exists: True


In [12]:

import os

graphviz_path = r"C:\Program Files (x86)\Graphviz\bin"
os.environ["PATH"] = graphviz_path + os.pathsep + os.environ.get("PATH", "")

import shutil
print("Graphviz:", shutil.which("dot"))


Graphviz: C:\Program Files (x86)\Graphviz\bin\dot.EXE


In [14]:
model = build_cnn(
    input_shape=config.INPUT_SHAPE,
    num_classes=config.NUM_CLASSES,
    learning_rate=config.LEARNING_RATE
)

show_architecture(model)

Model: "BrainScan_CNN"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_4 (Conv2D)           (None, 224, 224, 32)      896       
                                                                 
 batch_normalization_4 (Bat  (None, 224, 224, 32)      128       
 chNormalization)                                                
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 112, 112, 32)      0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 112, 112, 64)      18496     
                                                                 
 batch_normalization_5 (Bat  (None, 112, 112, 64)      256       
 chNormalization)                                                
                                                     

# Save prepared data for Kaggle

In [15]:
np.savez_compressed(
    "../data/processed/brainscan_data.npz",
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
    class_names=np.array(class_names)
)